## Pinecone Vector DB

Note: 
- Here, we create a index on pinecone with the name "rag-index". You can change it if you want. We also specify the dimension of the index to be 1024, which is the same as the dimension of the embeddings we are using. If the index already exists, we skip the creation step.

- The index is vector database where we will store our vectors

- Difference between vector store and vector database is that vector store is a wrapper around vector database which provides additional functionality like adding documents, similarity search etc. Vector database is the underlying database where the vectors are stored.

- Pinecone is a vector database that provides a managed service for storing and querying vectors. It provides a simple API for creating indexes, adding vectors, and querying vectors. It also provides features like automatic scaling, replication, and backup.

- We will use Pinecone as our vector database to store the vectors generated by the OpenAI embeddings model. We will create an index in Pinecone and then use the PineconeVectorStore from langchain_pinecone to interact with the index.

- We will add documents to the vector store and then perform similarity search to retrieve relevant documents based on a query.

- We will also create a retriever from the vector store which can be used in a RAG pipeline to retrieve relevant documents based on a query.

- Finally, we will use the retrieved documents to generate a response using a language model.

In [1]:
# Create your index and apikey from here https://www.pinecone.io/
import os
api_key = os.getenv("PINECONE_API_KEY")

In [2]:
# !pip install -qU langchain langchain-pinecone langchain-openai

In [3]:
from pinecone import Pinecone
pc = Pinecone(api_key=api_key)

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                              dimensions=1024,
                              api_key=os.getenv("OPENAI_API_KEY"))

c:\Users\mani2\Documents\RAG-Mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pinecone import ServerlessSpec

index_name = "rag-index"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [6]:
index

In [7]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [8]:
vector_store

In [9]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [10]:
vector_store.add_documents(documents=documents)

['1478b0d7-041f-4951-9769-bfcfe287ea77',
 '824f026a-7541-43d7-991f-ac8b98b7e4da',
 'c3a45ce1-9f2e-4576-a946-b24fcdc3879e',
 'a675fdd1-4e30-49d5-b073-50df614a367f',
 'b53c1748-61bb-4e85-a112-07a97252c51c',
 '2399e205-956a-450b-908c-aa605c0bcd25',
 '24866022-fc8d-40e1-854d-d3e78b1c8c20',
 '9debba22-16d3-42cf-9bba-457e9cd22ab2',
 '79baa089-f889-4e88-a239-478990b44d94',
 '4bcd8db9-aec1-4a91-bb06-9d78bc72f464']

In [11]:
# Query Directly
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [12]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.573013] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [13]:
# Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='a675fdd1-4e30-49d5-b073-50df614a367f', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

Miscellaneous features we can cover in Pinecone Vector DB::
- how to update and delete vectors from the index.
- how to use metadata to filter the vectors during retrieval.
- how to use Pinecone's built-in features like namespaces and collections to organize our vectors.
- how to use Pinecone's built-in features like namespaces and collections to organize our vectors.
- how to use Pinecone's built-in features like namespaces and collections to organize our vectors.